# Inverse Volatility Rank and Allocate to Future Contracts

**Reference:** Hands-On AI Trading with Python, QuantConnect, and AWS (2025)
Chapter 06 - Applied Machine Learning, Example 11 (p214)

## Objectifs

1. Charger un panier diversifié de contrats futures (indices, énergie, grains)
2. Prédire la volatilité d'ouverture avec une régression Ridge
3. Allouer les poids inversement proportionnels à la volatilité prédite
4. Comparer l'approche inverse-vol vs allocation équi-pondérée

> **[REFERENCE QC Cloud]**
> Ce notebook utilise QuantBook et nécessite l'environnement QuantConnect Cloud.
> Pour exécuter : https://www.quantconnect.com/research

In [ ]:
from AlgorithmImports import *
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

## 1. Sélection de l'univers Futures

Le livre Broad utilise 12 contrats futures couvrant 3 classes d'actifs :
- **Indices** : VIX, S&P 500 E-Mini, Nasdaq 100 E-Mini, DOW 30 E-Mini
- **Énergie** : Brent Crude, Gasoline, Heating Oil, Natural Gas
- **Grains** : Corn, Oats, Soybeans, Wheat

La diversification limite l'impact d'un seul actif sur le portefeuille global.

In [ ]:
# Define the futures universe
futures_tickers = {
    # Indices
    'VIX': Futures.Indices.VIX,
    'ES': Futures.Indices.SP_500_E_MINI,
    'NQ': Futures.Indices.NASDAQ_100_E_MINI,
    'YM': Futures.Indices.DOW_30_E_MINI,
    # Energy
    'BZ': Futures.Energy.BRENT_CRUDE,
    'RB': Futures.Energy.GASOLINE,
    'HO': Futures.Energy.HEATING_OIL,
    'NG': Futures.Energy.NATURAL_GAS,
    # Grains
    'ZC': Futures.Grains.CORN,
    'ZO': Futures.Grains.OATS,
    'ZS': Futures.Grains.SOYBEANS,
    'ZW': Futures.Grains.WHEAT,
}

print(f"Futures universe: {len(futures_tickers)} contracts")
for sector, tickers in [('Indices', ['VIX', 'ES', 'NQ', 'YM']),
                         ('Energy', ['BZ', 'RB', 'HO', 'NG']),
                         ('Grains', ['ZC', 'ZO', 'ZS', 'ZW'])]:
    print(f"  {sector}: {', '.join(tickers)}")

## 2. Chargement des données futures

On charge 2 ans de données quotidiennes pour les contrats front-month.
Les données incluent open, high, low, close et open interest.

In [ ]:
qb = QuantBook()

# Add futures to the book
future_symbols = {}
for name, ticker in futures_tickers.items():
    future = qb.add_future(ticker, extended_market_hours=True)
    future.set_filter(lambda universe: universe.front_month())
    future_symbols[name] = future.symbol

# Request history
end_date = datetime(2024, 1, 1)
start_date = end_date - timedelta(2 * 365)

# Get close price history
symbols = list(future_symbols.values())
history = qb.history(symbols, start_date, end_date, Resolution.DAILY)

if not history.empty:
    prices = history['close'].unstack(0)
    prices.dropna(axis=1, how='all', inplace=True)
    print(f"Price data shape: {prices.shape}")
    print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
    prices.head()
else:
    print("No history data returned. Check QC Cloud connectivity.")
    prices = pd.DataFrame()

## 3. Feature Engineering: Facteurs de volatilité

Le modèle Ridge utilise 3 facteurs par contrat :
- **Écart-type des rendements quotidiens** sur 60 jours (volatilité clôture)
- **ATR (Average True Range)** sur 60 jours (volatilité intraday)
- **Open Interest** (nombre de contrats ouverts)

Le label prédit est la volatilité des prix d'ouverture sur les 6 jours suivants.

In [ ]:
# Compute features per contract
features_by_contract = {}

for name in futures_tickers.keys():
    if name not in future_symbols:
        continue
    sym = future_symbols[name]
    
    if prices.empty or sym not in prices.columns:
        continue
    
    close = prices[sym].dropna()
    if len(close) < 60:
        continue
    
    returns = close.pct_change().dropna()
    
    # Feature 1: Rolling std of returns (60-day)
    std_60 = returns.rolling(60).std() * np.sqrt(252)
    
    # Feature 2: Rolling std at 3-month lookback
    std_90 = returns.rolling(90).std() * np.sqrt(252)
    
    # Feature 3: Rolling std at 6-month lookback
    std_180 = returns.rolling(180).std() * np.sqrt(252)
    
    # Label: Future volatility (6-day rolling std)
    future_vol = returns.rolling(6).std().shift(-6) * np.sqrt(252)
    
    features_by_contract[name] = pd.DataFrame({
        'std_60': std_60,
        'std_90': std_90,
        'std_180': std_180,
        'future_vol': future_vol,
    }).dropna()

print(f"Features computed for {len(features_by_contract)} contracts")
for name, feat_df in features_by_contract.items():
    print(f"  {name}: {len(feat_df)} samples")

## 4. Régression Ridge: Prédiction de volatilité

La régression Ridge est choisie car :
1. Elle gère efficacement la multicolinéarité (features corrélées)
2. Le terme de régularisation prévient le surapprentissage
3. Simple et efficace computationnellement

On entraîne un modèle par contrat futures.

In [ ]:
# Train Ridge model per contract and evaluate
from sklearn.metrics import r2_score, mean_absolute_error

results = []
predictions = {}

for name, feat_df in features_by_contract.items():
    if len(feat_df) < 60:
        continue
    
    feature_cols = ['std_60', 'std_90', 'std_180']
    X = feat_df[feature_cols].values
    y = feat_df['future_vol'].values
    
    # Train/test split (chronological)
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    model = Ridge(alpha=1.0)
    model.fit(X_train_s, y_train)
    
    y_pred = model.predict(X_test_s)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    results.append({
        'contract': name,
        'r2': r2,
        'mae': mae,
        'n_samples': len(feat_df),
        'coefficients': dict(zip(feature_cols, model.coef_)),
    })
    predictions[name] = {'actual': y_test, 'predicted': y_pred}

results_df = pd.DataFrame(results)
print("Ridge Regression Results per Contract:")
print(results_df[['contract', 'r2', 'mae', 'n_samples']].to_string(index=False))

## 5. Allocation Inverse-Volatilité

La formule d'allocation du livre Broad :

```
weight = multiplier / sigma / sum(sigma) / contract_multiplier
```

Les actifs moins volatils reçoivent un poids plus élevé, ce qui :
- Réduit le risque global du portefeuille
- Améliore le ratio de Sharpe
- Assure une diversification naturelle

In [ ]:
# Simulate inverse-volatility allocation vs equal-weight
weight_multiplier = 3.0  # From Broad book

# Get predicted volatilities (last prediction for each contract)
predicted_vols = {}
for name, pred in predictions.items():
    predicted_vols[name] = pred['predicted'][-1]  # Latest prediction

if predicted_vols:
    # Inverse-vol weights
    inv_vols = {k: 1/v for k, v in predicted_vols.items() if v > 0}
    vol_sum = sum(inv_vols.values())
    inv_weights = {k: weight_multiplier * v / vol_sum for k, v in inv_vols.items()}
    
    # Equal weights
    n_assets = len(predicted_vols)
    eq_weights = {k: 1.0 / n_assets for k in predicted_vols.keys()}
    
    # Compare allocations
    comparison = pd.DataFrame({
        'Predicted Vol': predicted_vols,
        'Inv-Vol Weight': inv_weights,
        'Equal Weight': eq_weights,
    }).round(4)
    
    print("Portfolio Allocation Comparison:")
    print(comparison.to_string())
    print(f"\nTotal inv-vol weight: {sum(inv_weights.values()):.2%}")
    print(f"Total equal weight:   {sum(eq_weights.values()):.2%}")
else:
    print("No predictions available. Check data loading.")

## 6. Sensibilité du Lookback

Le livre Broad montre que le Sharpe ratio est :
- **Le plus élevé** quand au moins un indicateur utilise un lookback de 6 mois
- **Le plus bas** quand tous les indicateurs utilisent 3 mois

On teste cette observation.

In [ ]:
# Lookback sensitivity analysis
lookback_configs = [
    ('3m only', [66, 66, 66]),
    ('3m+6m',  [66, 66, 132]),
    ('6m only', [132, 132, 132]),
    ('3m+6m+12m', [66, 132, 264]),
]

sensitivity_results = []

for config_name, lookbacks in lookback_configs:
    r2_values = []
    for name, feat_df in features_by_contract.items():
        if len(feat_df) < max(lookbacks) + 10:
            continue
        
        close = prices[future_symbols[name]].dropna()
        returns = close.pct_change().dropna()
        
        features = pd.DataFrame({
            f'std_{lb}': returns.rolling(lb).std() * np.sqrt(252)
            for lb in lookbacks
        })
        features['future_vol'] = returns.rolling(6).std().shift(-6) * np.sqrt(252)
        features.dropna(inplace=True)
        
        if len(features) < 40:
            continue
        
        X = features.drop('future_vol', axis=1).values
        y = features['future_vol'].values
        
        split = int(len(X) * 0.8)
        model = Ridge(alpha=1.0)
        model.fit(X[:split], y[:split])
        y_pred = model.predict(X[split:])
        r2_values.append(r2_score(y[split:], y_pred))
    
    avg_r2 = np.mean(r2_values) if r2_values else 0
    sensitivity_results.append({
        'config': config_name,
        'avg_r2': avg_r2,
        'n_contracts': len(r2_values),
    })

sensitivity_df = pd.DataFrame(sensitivity_results)
print("Lookback Sensitivity Analysis:")
print(sensitivity_df.to_string(index=False))

## 7. Conclusion

| Aspect | Observation |
|--------|-------------|
| Ridge pour vol | Gère multicolinéarité, efficace, régularisé |
| Allocation inv-vol | Plus de poids aux actifs stables, meilleur Sharpe |
| Lookback optimal | 6 mois donne les meilleurs résultats (Broad Ch6 p217) |
| Diversification | 3 classes d'actifs (indices, énergie, grains) |

**Points clés du livre Broad (Ch6 Ex11)** :
- Le Sharpe est typiquement le plus élevé avec au moins un lookback de 6 mois
- Le multiplicateur de poids (3x) équilibre exposition et appels de marge
- Les contrats à faible volatilité dominent l'allocation naturellement

**Stratégie complète** : voir `main.py` pour l'implémentation algorithmique QC Cloud avec gestion des rollovers et stop-loss.